In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from statsmodels.tsa.deterministic import DeterministicProcess, CalendarFourier
import warnings
import os

warnings.filterwarnings('ignore')

# 1. Load Data
def load_data():
    print("Loading data...")
    base_path = '/kaggle/input/store-sales-time-series-forecasting/'
    if not os.path.exists(base_path + 'train.csv'):
        base_path = '/kaggle/input/competitions/store-sales-time-series-forecasting/'
        
    train = pd.read_csv(base_path + 'train.csv', parse_dates=['date'])
    test = pd.read_csv(base_path + 'test.csv', parse_dates=['date'])
    stores = pd.read_csv(base_path + 'stores.csv')
    oil = pd.read_csv(base_path + 'oil.csv', parse_dates=['date'])
    holidays = pd.read_csv(base_path + 'holidays_events.csv', parse_dates=['date'])
    return train, test, stores, oil, holidays

# 2. Granular Feature Engineering
def prepare_features(train, test, stores, oil, holidays):
    print("Preparing features...")
    
    # Pre-rename columns to avoid collisions
    stores = stores.rename(columns={'type': 'store_type'})
    holidays = holidays.rename(columns={'type': 'holiday_type'})
    
    # Oil processing
    oil['date'] = pd.to_datetime(oil['date'])
    oil = oil.set_index('date').resample('D').mean().interpolate(method='linear').reset_index()
    for w in [1, 3, 7, 14]:
        oil[f'oil_lags_{w}'] = oil['dcoilwtico'].shift(w)
        oil[f'oil_rolling_{w}'] = oil['dcoilwtico'].rolling(w).mean()
    
    # Holidays processing
    holidays = holidays[holidays['transferred'] == False]
    
    # Combine train and test
    df = pd.concat([train, test], axis=0)
    df['date'] = pd.to_datetime(df['date'])
    
    # Merge datasets
    df = df.merge(stores, on='store_nbr', how='left')
    df = df.merge(oil, on='date', how='left')
    
    # Merge holidays by locale
    nat_hol = holidays[holidays['locale'] == 'National'].drop_duplicates('date')
    df = df.merge(nat_hol[['date', 'holiday_type']], on='date', how='left').rename(columns={'holiday_type': 'nat_hol_type'})
    
    reg_hol = holidays[holidays['locale'] == 'Regional'].rename(columns={'locale_name': 'state'})
    df = df.merge(reg_hol[['date', 'state', 'holiday_type']], on=['date', 'state'], how='left').rename(columns={'holiday_type': 'reg_hol_type'})
    
    loc_hol = holidays[holidays['locale'] == 'Local'].rename(columns={'locale_name': 'city'})
    df = df.merge(loc_hol[['date', 'city', 'holiday_type']], on=['date', 'city'], how='left').rename(columns={'holiday_type': 'loc_hol_type'})
    
    # Promotion Features
    df['onpromotion'] = df['onpromotion'].fillna(0)
    for w in [7, 14, 28]:
        df[f'promo_rolling_{w}'] = df.groupby(['store_nbr', 'family'])['onpromotion'].transform(lambda x: x.rolling(w).mean())
    
    # Time features
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_payday'] = df['day_of_month'].isin([15, 30, 31]).astype(int)
    df['newyear'] = (df['date'].dt.dayofyear == 1).astype(int)
    
    # Categorical encoding (Keep original columns for merging)
    le = LabelEncoder()
    cat_cols = ['city', 'state', 'store_type', 'cluster', 'nat_hol_type', 'reg_hol_type', 'loc_hol_type']
    for col in cat_cols:
        df[col] = le.fit_transform(df[col].astype(str))
    
    # Special handling for family: encode into a new column but keep original
    df['family_encoded'] = le.fit_transform(df['family'].astype(str))
        
    df['sales'] = np.log1p(df['sales'])
    return df

# 3. Boosted Hybrid Model
class BoostedHybrid:
    def __init__(self, model_1, model_2):
        self.model_1 = model_1
        self.model_2 = model_2
        self.y_columns = None

    def fit(self, X_1, X_2, y):
        self.model_1.fit(X_1, y)
        y_fit = pd.DataFrame(self.model_1.predict(X_1), index=X_1.index, columns=y.columns)
        y_resid = y - y_fit
        y_resid = y_resid.stack(['store_nbr', 'family'])
        self.model_2.fit(X_2, y_resid)
        self.y_columns = y.columns

    def predict(self, X_1, X_2):
        y_pred_1 = pd.DataFrame(self.model_1.predict(X_1), index=X_1.index, columns=self.y_columns)
        y_pred_1 = y_pred_1.stack(['store_nbr', 'family'])
        y_pred_2 = self.model_2.predict(X_2)
        y_pred = y_pred_1 + y_pred_2
        return y_pred.unstack(['store_nbr', 'family'])

# 4. Main Execution
def main():
    train, test, stores, oil, holidays = load_data()
    train = train[train['date'] >= '2017-01-01']
    df = prepare_features(train, test, stores, oil, holidays)
    
    # Pivot for Trend Model
    y = df[df['date'] < '2017-08-16'].set_index(['date', 'store_nbr', 'family'])['sales'].unstack(['store_nbr', 'family']).fillna(0)
    y.index = pd.to_datetime(y.index)
    y = y.asfreq('D')
    
    # Model 1 (Linear Trend + Fourier)
    fourier = CalendarFourier(freq="W", order=4)
    dp = DeterministicProcess(index=y.index, constant=True, order=1, additional_terms=[fourier], drop=True)
    X_1 = dp.in_sample()
    X_test_1 = dp.out_of_sample(steps=16)
    
    # Model 2 (LightGBM)
    # Add Lags
    for lag in [16, 17, 18, 19, 20, 21, 22, 28, 30]:
        df[f'lag_{lag}'] = df.groupby(['store_nbr', 'family'])['sales'].shift(lag)
        
    excluded = ['id', 'date', 'sales', 'dcoilwtico', 'family'] # Exclude original 'family' string
    features_2 = [col for col in df.columns if col not in excluded]
    
    # Prepare X_2 and X_test_2
    X_2_full = df[df['date'] < '2017-08-16'].sort_values(['date', 'store_nbr', 'family'])
    X_test_2_full = df[df['date'] >= '2017-08-16'].sort_values(['date', 'store_nbr', 'family'])
    
    X_2 = X_2_full[features_2].fillna(0)
    X_test_2 = X_test_2_full[features_2].fillna(0)
    
    # Ensure y is sorted the same way
    y = y.sort_index()
    
    seeds = [42, 123, 777]
    final_preds = None
    
    for seed in seeds:
        print(f"Training hybrid model with seed {seed}...")
        model_1 = Ridge(alpha=0.7)
        model_2 = lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.03, num_leaves=63, max_depth=10, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5, random_state=seed, verbosity=-1)
        
        hybrid = BoostedHybrid(model_1, model_2)
        hybrid.fit(X_1, X_2, y)
        
        y_pred = hybrid.predict(X_test_1, X_test_2)
        if final_preds is None:
            final_preds = y_pred / len(seeds)
        else:
            final_preds += y_pred / len(seeds)
            
    # CRITICAL: During stack/unstack, family names are preserved as strings in the index
    y_submit = final_preds.stack(['store_nbr', 'family']).reset_index()
    y_submit.columns = ['date', 'store_nbr', 'family', 'sales']
    
    # Ensure types match before merge
    y_submit['family'] = y_submit['family'].astype(str)
    test['family'] = test['family'].astype(str)
    y_submit['store_nbr'] = y_submit['store_nbr'].astype(int)
    test['store_nbr'] = test['store_nbr'].astype(int)
    
    submission = test.merge(y_submit, on=['date', 'store_nbr', 'family'], how='left')
    submission['sales'] = np.expm1(submission['sales']).clip(0, None)
    submission.loc[submission['date'].dt.dayofyear == 1, 'sales'] = 0
    submission[['id', 'sales']].to_csv('submission.csv', index=False)
    print("Success! Final submission saved to submission.csv")

if __name__ == "__main__":
    main()


Loading data...
Preparing features...
Training hybrid model with seed 42...
Training hybrid model with seed 123...
Training hybrid model with seed 777...
Success! Final submission saved to submission.csv
